# Secure Channel: build HTTPS by hand

**The story.** When you open your bank's website, your password travels across a network that strangers can watch, yet only the bank can read it. That is HTTPS. Today you build the same thing by hand.

- **You are the client** (the browser).
- **The facilitator is the server** (the website).
- **The public GitHub repo is the internet**: everyone can see every file that travels.

You will send honest feedback about the sessions to the facilitator. Everyone can see the file you send. Only the facilitator can read it.

We build the channel in two stages and break the first one on purpose.

| Stage | Idea | Who can read your message |
|---|---|---|
| 1 | One personal key, made from your name | You, the facilitator, and anyone who can guess how the key is made |
| 2 | A random session key, locked with the facilitator's public key | You and the facilitator only |

Work in pairs. For every **Predict** line, say your answer aloud before you run the cell.

## Part 0: Setup

Run this once. It installs the industry library that does the real encryption.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import secure_channel as sc

class_message = sc.read_class_message()
print(class_message)

## Part 1: Your personal key

Everyone downloaded the same class message. You add one thing that is yours: your name.

**Predict:** will you and your partner get the same key? Will you get the same key if you run the cell twice?

In [ ]:
MY_NAME = input("Your full name: ")

my_key = sc.make_key(MY_NAME)
print("Your key:", my_key)
print("Your id: ", sc.student_id(my_key))

**Predict:** you change only the capital letters and spaces in your name. Same key or a new key? What if you change one letter?

In [ ]:
print(sc.make_key(MY_NAME.upper() + "   ") == my_key)
print(sc.make_key(MY_NAME + "x"))

### Look inside `make_key`

No magic: it is three steps you can do by hand.

1. Clean the name (lower case, single spaces).
2. Mix the name with the class message using **HMAC-SHA256**. A hash turns any text into a fixed 32 bytes. Change one letter and every byte changes.
3. Write the 32 bytes as text using **Base64**, which is an encoding like hex, only shorter.

In [ ]:
import base64, hashlib, hmac

clean = " ".join(MY_NAME.lower().split())
raw = hmac.new(class_message.encode(), clean.encode(), hashlib.sha256).digest()

print("Clean name:", clean)
print("Bytes:     ", len(raw))
print("As hex:    ", raw.hex())
print("As Base64: ", base64.urlsafe_b64encode(raw).decode())
print("Same as make_key?", base64.urlsafe_b64encode(raw).decode() == my_key)

## Part 2: Stage 1, one key locks and unlocks

This is **symmetric** encryption: the same key on both sides. The facilitator can make your key too, because the facilitator knows your name and the class message.

In [ ]:
locked = sc.encrypt("The pace today was too fast for me.", my_key)
print(locked)

In [ ]:
print(sc.decrypt(locked, my_key))

**Predict:** what happens if you unlock with a key that is wrong by one name?

In [ ]:
wrong_key = sc.make_key("Somebody Else")
try:
    print(sc.decrypt(locked, wrong_key))
except ValueError as error:
    print("Could not unlock:", error)

### Your turn: write your own `my_encrypt` and `my_decrypt`

Use XOR from the last session. Each character of the message is XORed with one character of the key; when the key runs out, start it again (`i % len(key)`). Store the result as hex.

Fill in the two functions, then run the check cell.

In [ ]:
def my_encrypt(message, key):
    """Return the message XORed with the key, as hex text like '1f 0a 33'."""
    # your code here


def my_decrypt(hex_text, key):
    """Turn the hex text back into the message."""
    # your code here

In [ ]:
test = "Python is fun"
result = my_encrypt(test, my_key)
if result is None:
    print("Not written yet. Fill in my_encrypt and my_decrypt above.")
elif my_decrypt(result, my_key) == test and result != test:
    print("It works:", result)
else:
    print("Not yet. I got back:", my_decrypt(result, my_key))

### Break it

Ask your partner for a locked message (they run `sc.encrypt(...)` and send you the text). Do **not** ask for their key.

**Predict:** can you read it?

In [ ]:
partner_name = input("Your partner's full name: ")
partner_locked = input("Paste their locked message: ")

partner_key = sc.make_key(partner_name)
try:
    print("You read:", sc.decrypt(partner_locked, partner_key))
except ValueError as error:
    print("Could not unlock:", error)

**What went wrong?** The key was made only from things other people know: the class message (it is on GitHub) and your name. A key is only as secret as the most secret thing that went into it.

Stage 2 fixes this with two changes:

- Your real key becomes **random**. Nobody can rebuild it from your name.
- You never send that key in the open. You lock it with the facilitator's **public key**. A public key can only lock; the matching **private key**, which stays on the facilitator's laptop, is the only thing that unlocks.

This is **hybrid** encryption. Your browser does the same thing every time you see the lock icon on a website.

## Part 3: Stage 2, seal your feedback

First check that you hold the real public key. Compare this code with the one the facilitator shows on the screen. If it is different, stop and tell the facilitator.

In [ ]:
print("Public key fingerprint:", sc.public_key_fingerprint())

Now make your random session key. It is saved in `my_secrets/` on your laptop.

**Predict:** will your partner's session key have anything in common with yours?

In [ ]:
session_key = sc.get_session_key()
print("Session key saved in:", sc.SESSION_KEY_FILE)

Write your real feedback. Be honest: what helped, what was too fast, what you want more of.

In [ ]:
my_feedback = input("Your feedback for the facilitator: ")

envelope = sc.seal(my_feedback, MY_NAME)
path = sc.save_envelope(envelope)
print("Saved:", path.name)
envelope

Look at the envelope. It has two locked parts:

- `token`: your message, locked with your session key (symmetric).
- `wrapped_key`: your session key, locked with the facilitator's public key (asymmetric).

**Predict:** your partner knows your name, your personal key and this whole file. Can they read it?

In [ ]:
personal_key = sc.make_key(MY_NAME)
try:
    print(sc.decrypt(envelope["token"], personal_key))
except ValueError as error:
    print("The personal key cannot open it:", error)

## Part 4: Send it through GitHub

Open a terminal in the repo folder and run one line at a time. Replace `<YOURNAME>`.

```cmd
git checkout -b feedback-<YOURNAME>
git status
```

**Look at `git status` carefully.** Your new file in `feedback/` is listed. The `my_secrets` folder is **not**. Open `.gitignore` at the top of the repo to see why. Secrets never go to GitHub.

```cmd
git add SecureChannel/feedback
git commit -m "Add sealed feedback"
git push -u origin feedback-<YOURNAME>
```

Then open your fork on GitHub and click **Compare & pull request**, then **Create pull request**.

## Part 5: Read the facilitator's reply

After the facilitator answers, get the new files and run the cell.

```cmd
git checkout main
git pull upstream main
```

The reply is locked with **your** session key, which only you and the facilitator hold.

In [ ]:
replies = sc.open_replies(MY_NAME)
if not replies:
    print("No reply yet.")
for sent_at, text in replies:
    print(sent_at, "->", text)

## Part 6: You just did an HTTPS handshake

Every step you ran has a twin in HTTPS. Your browser does all of this in a fraction of a second, every time you see the lock icon.

| Step in HTTPS | What the browser and website do | What you did |
|---|---|---|
| 1. Hello | Browser asks the website for its certificate | You pulled `teacher_public_key.pem` from the repo |
| 2. Check the certificate | Browser checks the public key really belongs to the website | You compared the fingerprint with the screen (Part 3) |
| 3. Key exchange | Browser and website agree on a secret session key | You made a random session key and sent it as `wrapped_key` |
| 4. Encrypted request | Browser sends your data locked with the session key | Your feedback travelled as `token` |
| 5. Encrypted response | Website answers, locked with the same session key | The facilitator's reply (Part 5) |

Steps 1 to 3 are **asymmetric** and slow. Steps 4 and 5 are **symmetric** and fast. That is why HTTPS uses both.

**Where real HTTPS is stronger than our lab**

- A browser does not check fingerprints by eye. The certificate is signed by a trusted company (a Certificate Authority) and the check is automatic.
- A browser makes a **new** session key for every connection. We reuse one.
- Our lab sends the session key wrapped with the server's public key, which is how HTTPS worked for many years. Today's version (TLS 1.3) lets both sides *build* the key together without ever sending it, and the server uses its private key only to prove who it is. The gain: even if the server's private key is stolen later, old conversations stay locked.

### See the real thing

This cell opens a real HTTPS connection to GitHub and prints what was agreed.

**Predict:** which symmetric cipher from this lab's family will appear in the output? (Hint: we met AES at the end of the last session.)

In [ ]:
import socket, ssl

host = "github.com"
try:
    context = ssl.create_default_context()
    with socket.create_connection((host, 443), timeout=10) as sock:
        with context.wrap_socket(sock, server_hostname=host) as tls:
            name, version, bits = tls.cipher()
            cert = tls.getpeercert()
            print("Protocol:        ", version)
            print("Session cipher:  ", name, f"({bits}-bit key)")
            print("Certificate for: ", dict(item[0] for item in cert["subject"])["commonName"])
            print("Signed by:       ", dict(item[0] for item in cert["issuer"])["organizationName"])
            print("Valid until:     ", cert["notAfter"])
except OSError as error:
    print("Could not connect:", error)

## What you built

| Word | What it means here |
|---|---|
| Hash | `make_key` turned your name into 32 bytes that cannot be turned back |
| Symmetric | One key locks and unlocks (Stage 1, and the `token`) |
| Asymmetric | Public key locks, private key unlocks (the `wrapped_key`) |
| Hybrid | Both together: fast symmetric lock for the message, asymmetric lock for the key |
| HTTPS | Hybrid encryption between a client and a server, plus a certificate check |
| `.gitignore` | The list of files Git must never upload |

**Think about it**

1. If you delete `my_secrets/`, what can you no longer read? What can the facilitator still read?
2. Anyone can type your name into Part 3 and send feedback as you. What would the facilitator need from you to be sure it was you? (This is called a digital signature.)
3. Everyone can see *that* you sent a file. What is still hidden?